In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd

from heart_disease.data.ingestion import load_file
from heart_disease.data.splitting import split_data
import heart_disease.models.train as t
from heart_disease.features.preprocessing import (
    create_training_pipeline,
    create_categorical_pipeline,
    create_numeric_pipeline,
    create_preprocessor,
)
from heart_disease.config import (
    INTERIM_DATA_DIR,
    RANDOM_STATE,
    TARGET_COLUMN,
    FEATURES,
    MODEL_DIR,
)

# Import and Split Data

In [60]:
df = load_file(INTERIM_DATA_DIR / "cleaned_dataset.csv")
df.head()

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
0,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,1
2,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [41]:
X_train, X_test, y_train, y_test = split_data(df, TARGET_COLUMN, FEATURES)
X_test.to_parquet(INTERIM_DATA_DIR / "X_test.parquet", index=False)
y_test.to_frame("target").to_parquet(INTERIM_DATA_DIR / "y_test.parquet", index=False)

2026-09-06 18:23:06,989 | INFO | heart_disease.data.splitting | Created with shape of X_train: (736, 13), X_test: (184, 13), y_train: (736,), y_test: (184,)


# Build Baseline Model

In [42]:
logistic_regression = create_training_pipeline(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
)
t.cross_validate_model(logistic_regression, X_train, y_train, t.TrainingConfig())

2026-09-06 18:23:07,067 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:09,165 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425


# Evaluate Dataset Based On Model Performance

## Evaluate maximum missing data allowed in each observation

Each observation contains fourteen features that may provide useful predictive information.
However, some observation have more than half of their features missing, potentially indicating poor data quality. 
Therefore, we evaluate different threshold for removing observations based on their proportion of missing values using cross-validation. 
The selected threshold must satisfy two constraint: the proportion of discarded observations must not exceed `ten percent` of the dataset, and the refined dataset must have less than one percent difference in target mean with the original dataset.

In [43]:
results = []

for num in range(6, 14):
    initial_rows = X_train.shape[0]
    temporary_df = X_train.copy()
    temporary_df[TARGET_COLUMN] = y_train.copy()
    temporary_df = temporary_df.dropna(thresh=num)

    X = temporary_df[FEATURES]
    y = temporary_df[TARGET_COLUMN]

    scores = t.cross_validate_model(logistic_regression, X, y, t.TrainingConfig())

    scores["threshold"] = num
    scores["total_rows"] = X.shape[0]
    scores["%_rows_deleted"] = (initial_rows - X.shape[0]) / initial_rows * 100
    scores["total_rows x mean"] = scores["total_rows"] * scores["mean"]
    scores["target_mean"] = y.mean()
    results.append(scores)

results = pd.concat(results)
results = results.sort_values("total_rows x mean", ascending=False).set_index(
    "threshold"
)
results

2026-09-06 18:23:09,336 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1


2026-09-06 18:23:11,911 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011
2026-09-06 18:23:11,926 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:13,319 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.844 ± 0.019
2026-09-06 18:23:13,338 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:14,741 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.852 ± 0.034
2026-09-06 18:23:14,783 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:16,846 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,total_rows,%_rows_deleted,total_rows x mean,target_mean
threshold,,,,,,,,,,,
7,0.817073,0.857143,0.825000,0.853503,0.867470,0.844038,0.019493,727,1.222826,613.615494,0.551582
6,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425,736,0.000000,612.584184,0.552989
8,0.800000,0.872483,0.887417,0.825000,0.877419,0.852464,0.033918,690,6.250000,588.200132,0.546377
9,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627,688,6.521739,586.914157,0.545058
10,0.870130,0.829630,0.828571,0.861314,0.819444,0.841818,0.020031,649,11.820652,546.339784,0.546995
11,0.850000,0.877193,0.904348,0.873950,0.833333,0.867765,0.024358,474,35.597826,411.320489,0.613924
12,0.869565,0.857143,0.868687,0.888889,0.844444,0.865746,0.014742,401,45.516304,347.164008,0.586035
13,0.808511,0.800000,0.840000,0.830189,0.875000,0.830740,0.026397,262,64.402174,217.653844,0.473282


Based on the results of testing the multiple threshold above, a threshold of twelve, which retains only rows with at least twelve non-missing features, results in the highest F1-score. However, this threshold has tow major drawbacks: it discards thirty-six percent of the training data and results in a substantial difference in the target mean compared with the original dataset.
Three threshold satisfy our previously defined constraints: seven, eight, and nine while six represent the original dataset. Amongthese feasible threshold, we choose a threshold of nine because it achieves a hogher F1-score while stll satisfying both constrains.

In [44]:
len(X_train.columns)

13

The threshold selection process is based on `X_train` with `y_train` and depends on the total number of features. Therefore, the threshold values should be adjusted according to the dimensionality of the dataset.


In [45]:
mask = X_train.notna().sum(axis=1) >= 8

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]

In [46]:
X_train.shape

(688, 13)

In [47]:
strategies = {
    "indicators_both": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=True),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_numeric": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_categorical": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
    "indicators_none": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
}

In [48]:
list_of_scores = []

for name, strategy in strategies.items():
    pipeline = create_training_pipeline(
        model=LogisticRegression(random_state=RANDOM_STATE), preprocessor=strategy
    )

    scores = t.cross_validate_model(pipeline, X_train, y_train, t.TrainingConfig())

    scores["strategy"] = name
    list_of_scores.append(scores)

list_of_scores = (
    pd.concat(list_of_scores).sort_values("mean", ascending=False).set_index("strategy")
)
list_of_scores

2026-09-06 18:23:21,618 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1


2026-09-06 18:23:23,699 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.853 ± 0.024
2026-09-06 18:23:23,702 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:24,609 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.858 ± 0.021
2026-09-06 18:23:24,637 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:26,097 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.829 ± 0.017
2026-09-06 18:23:26,107 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:23:27,533 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
strategy,,,,,,,
indicators_numeric,0.845638,0.860759,0.826667,0.887324,0.870130,0.858104,0.020733
indicators_both,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627
indicators_categorical,0.826667,0.828025,0.800000,0.834532,0.853503,0.828546,0.017189
indicators_none,0.826667,0.828025,0.800000,0.834532,0.853503,0.828546,0.017189


# Evaluate Different Algorithm 

In [49]:
list_of_models = [
    LogisticRegression(random_state=42),
    RandomForestClassifier(random_state=42),
    CalibratedClassifierCV(SVC(), ensemble=False),
    GradientBoostingClassifier(random_state=42),
    KNeighborsClassifier(),
    XGBClassifier(random_state=42),
]

In [50]:
t.model_comparison_cv(
    list_of_models, create_training_pipeline, X_train, y_train, t.TrainingConfig()
)

,Model,fit_time,Accuracy,Precision,Recall,F1,ROC AUC
0,LogisticRegression,0.3686,0.8402,0.8543,0.8533,0.8531,0.9106
1,CalibratedClassifierCV,0.3853,0.8329,0.8310,0.8720,0.8502,0.9026
2,GradientBoostingClassifier,0.8276,0.8256,0.8359,0.8480,0.8415,0.8855
3,RandomForestClassifier,1.0804,0.8257,0.8404,0.8427,0.8406,0.8879
4,XGBClassifier,2.5570,0.8198,0.8211,0.8560,0.8378,0.8727
5,KNeighborsClassifier,0.0878,0.8169,0.8294,0.8400,0.8330,0.8690


# Tune Hyperparameters for Selected Algorithm

In [51]:
param_grid = [
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["lbfgs"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["saga"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__l1_ratio": [0, 0.5, 1],
    },
]

In [52]:
search = t.grid_search(
    logistic_regression, param_grid, X_train, y_train, t.TrainingConfig()
)

2026-09-06 18:24:00,845 | INFO | heart_disease.models.train | Starting grid search


2026-09-06 18:24:45,930 | INFO | heart_disease.models.train | Best parameters: {'classifier__C': 1, 'classifier__l1_ratio': 0.5, 'classifier__max_iter': 5000, 'classifier__solver': 'saga'} | Best CV scores: 0.854


In [53]:
search.best_score_

np.float64(0.854310694099764)

In [54]:
best_model = create_training_pipeline(
    LogisticRegression(
        max_iter=5000, l1_ratio=0.5, C=1, solver="saga", random_state=RANDOM_STATE
    )
)
t.cross_validate_model(best_model, X_train, y_train, t.TrainingConfig())

2026-09-06 18:24:46,016 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:24:51,839 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.854 ± 0.022


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.843537,0.858974,0.818792,0.867133,0.883117,0.854311,0.02188


In [55]:
logistic_regression = create_training_pipeline(
    LogisticRegression(random_state=RANDOM_STATE)
)
t.cross_validate_model(logistic_regression, X_train, y_train, t.TrainingConfig())

2026-09-06 18:24:51,892 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-09-06 18:24:53,365 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.853 ± 0.024


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627


In [56]:
logistic_regression = t.train_model(best_model, X_train, y_train)

2026-09-06 18:24:55,842 | INFO | heart_disease.models.train | Training: LogisticRegression with shape of dataframe: (688, 13)


In [57]:
logistic_regression.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'preprocessor', 'classifier', 'preprocessor__n_jobs', 'preprocessor__remainder', 'preprocessor__sparse_threshold', 'preprocessor__transformer_weights', 'preprocessor__transformers', 'preprocessor__verbose', 'preprocessor__verbose_feature_names_out', 'preprocessor__numeric', 'preprocessor__categorical', 'preprocessor__numeric__memory', 'preprocessor__numeric__steps', 'preprocessor__numeric__transform_input', 'preprocessor__numeric__verbose', 'preprocessor__numeric__imputer', 'preprocessor__numeric__scaler', 'preprocessor__numeric__imputer__add_indicator', 'preprocessor__numeric__imputer__copy', 'preprocessor__numeric__imputer__fill_value', 'preprocessor__numeric__imputer__keep_empty_features', 'preprocessor__numeric__imputer__missing_values', 'preprocessor__numeric__imputer__strategy', 'preprocessor__numeric__scaler__copy', 'preprocessor__numeric__scaler__with_mean', 'preprocessor__numeric__scaler__with_std', 'preprocessor__cat

In [58]:
features_in = logistic_regression.named_steps["preprocessor"].get_feature_names_out()
print(features_in)
print(len(features_in))

['numeric__age' 'numeric__trestbps' 'numeric__chol' 'numeric__thalch'
 'numeric__oldpeak' 'numeric__missingindicator_trestbps'
 'numeric__missingindicator_chol' 'numeric__missingindicator_thalch'
 'numeric__missingindicator_oldpeak' 'categorical__sex_Female'
 'categorical__sex_Male' 'categorical__cp_asymptomatic'
 'categorical__cp_atypical angina' 'categorical__cp_non-anginal'
 'categorical__cp_typical angina' 'categorical__fbs_False'
 'categorical__fbs_True' 'categorical__restecg_lv hypertrophy'
 'categorical__restecg_normal' 'categorical__restecg_st-t abnormality'
 'categorical__exang_False' 'categorical__exang_True'
 'categorical__slope_downsloping' 'categorical__slope_flat'
 'categorical__slope_upsloping' 'categorical__ca_0.0'
 'categorical__ca_1.0' 'categorical__ca_2.0' 'categorical__ca_3.0'
 'categorical__thal_fixed defect' 'categorical__thal_normal'
 'categorical__thal_reversable defect'
 'categorical__missingindicator_fbs_False'
 'categorical__missingindicator_fbs_True'
 'categ

In [59]:
path = MODEL_DIR / "logistic_regression.joblib"
t.save_model(logistic_regression, path)

2026-09-06 18:24:56,026 | INFO | heart_disease.models.train | Saving LogisticRegression model to /home/irasionize/heart-disease-ml/models/logistic_regression.joblib
